# Waste Classification — Full Pipeline
## ViT vs ResNet50V2 vs MobileNetV2
**Authors:** Siyabonga Dunn (223017814) & Alunge Hlubi (223019985)  
**Supervisor:** Dr. Andronicus Ayobami Akinyelu — UKZN

### Pipeline Steps
1. Dataset download & 80/20 stratified split  
2. Preprocessing & augmentation functions  
3. Model builders (MobileNetV2, ResNet50V2, ViT)  
4. Phase 1 — 5-Fold Cross-Validation on 80% pool  
5. Phase 2 — Final training on full 80% pool  
6. Final test evaluation + McNemar's test  
7. Explainability — Attention Maps & Grad-CAM gallery


In [ ]:
# ============================================================
# WASTE CLASSIFICATION — FULL PIPELINE
# ViT vs ResNet50V2 vs MobileNetV2
# Authors : Siyabonga Dunn (223017814) & Alunge Hlubi (223019985)
# Supervisor: Dr. Andronicus Ayobami Akinyelu — UKZN
# ============================================================

: 

In [ ]:
# !pip install -q kagglehub tensorflow transformers torch scikit-learn seaborn statsmodels

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
import torch
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
from statsmodels.stats.contingency_tables import mcnemar
from transformers import ViTModel
from collections import defaultdict

warnings.filterwarnings('ignore')

# Fix seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("All imports successful")

---
## STEP 1: DATASET DOWNLOAD AND 80/20 STRATIFIED SPLIT

In [ ]:
IMG_SIZE    = 224
NUM_CLASSES = 6
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

print(f"Classes : {CLASS_NAMES}")
print(f"IMG_SIZE: {IMG_SIZE}x{IMG_SIZE}")

# ============================================================
# STEP 1: DATASET DOWNLOAD AND 80/20 STRATIFIED SPLIT
# ============================================================

In [ ]:
print("=" * 60)
print("STEP 1A: DOWNLOADING DATASET FROM KAGGLE")
print("=" * 60)

path     = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
DATA_DIR = os.path.join(path, "Garbage classification", "Garbage classification")

print(f"Dataset path: {DATA_DIR}")

# Verify structure and count images per class
print("\nVerifying dataset structure:")
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_DIR, cls)
    if os.path.exists(cls_dir):
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {cls:<12}: {n} images")
    else:
        print(f"  {cls:<12}: NOT FOUND — check folder name")

In [ ]:
print("\n" + "=" * 60)
print("STEP 1B: LOADING ALL IMAGE PATHS AND LABELS")
print("=" * 60)

file_paths, labels = [], []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_file in os.listdir(class_dir):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            # Store relative path (class_name/filename)
            file_paths.append(os.path.join(class_name, img_file))
            labels.append(class_idx)

print(f"Total images loaded: {len(file_paths)}")

---
## STEP 2: PREPROCESSING AND AUGMENTATION FUNCTIONS

In [ ]:
print("\n" + "=" * 60)
print("STEP 1C: 80/20 STRATIFIED SPLIT")
print("=" * 60)

train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

print(f"Training pool (80%): {len(train_paths)} images")
print(f"Test set     (20%): {len(test_paths)} images — ISOLATED, DO NOT TOUCH UNTIL STEP 4")

# Helper: print class distribution
def print_dist(label_list, name):
    unique, counts = np.unique(label_list, return_counts=True)
    print(f"\n{name} distribution:")
    for idx, count in zip(unique, counts):
        pct = count / len(label_list) * 100
        print(f"  {CLASS_NAMES[idx]:<12}: {count:>4} ({pct:.1f}%)")

print_dist(train_labels, "Training pool")
print_dist(test_labels,  "Test set (isolated)")

# Save isolated test set — load only in Step 4
with open('test_set_isolated.pkl', 'wb') as f:
    pickle.dump({'paths': test_paths, 'labels': test_labels}, f)
print("\nTest set saved to 'test_set_isolated.pkl' — do not load until Step 4")

# ============================================================
# STEP 2: PREPROCESSING AND AUGMENTATION FUNCTIONS
# ============================================================

---
## STEP 3A: CNN MODEL BUILDERS

In [ ]:
print("\n" + "=" * 60)
print("STEP 2: PREPROCESSING AND AUGMENTATION FUNCTIONS")
print("=" * 60)

def preprocess_image(image_path, label):
    """Load, decode, resize and normalise a single image to [0,1]."""
    full_path = tf.strings.join([DATA_DIR, "/", image_path])
    image     = tf.io.read_file(full_path)
    image     = tf.image.decode_jpeg(image, channels=3)
    image     = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image     = tf.cast(image, tf.float32) / 255.0
    return image, label


def augment_image(image, label):
    """
    Augmentation pipeline applied to training set only.
    Covers: rotation, flipping, zoom, brightness, contrast.
    Directly addresses the small 'trash' class (127 images).
    """
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    # Random zoom via crop-and-resize
    image = tf.image.random_crop(image, size=[int(IMG_SIZE * 0.85),
                                               int(IMG_SIZE * 0.85), 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def create_cnn_dataset(paths, label_list, batch_size, augment=False, shuffle=False):
    """
    Build a tf.data pipeline for CNN models.
    Labels are one-hot encoded (categorical_crossentropy).
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, label_list))
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (x, tf.one_hot(y, depth=NUM_CLASSES)))
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def create_vit_dataset(paths, label_list, batch_size, augment=False, shuffle=False):
    """
    Build a tf.data pipeline for the ViT model.
    Labels are integer class indices (SparseCategoricalCrossentropy).
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, label_list))
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (x, tf.cast(y, tf.int32)))
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("Preprocessing and augmentation functions ready")

# ============================================================
# STEP 3A: CNN MODEL BUILDERS
# ============================================================

---
## STEP 3B: ViT MODEL — LOAD PRETRAINED WEIGHTS FROM HUGGINGFACE

In [ ]:
print("\n" + "=" * 60)
print("STEP 3A: CNN MODEL BUILDERS")
print("=" * 60)

def build_mobilenetv2(num_classes=NUM_CLASSES, learning_rate=1e-4, weight_decay=1e-4):
    """
    MobileNetV2 with frozen ImageNet base and custom 6-class head.
    Lightweight CNN baseline — ~3.4M parameters.
    """
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False   # Phase 1: head only

    model = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(
            num_classes, activation='softmax',
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay)
        )
    ], name='MobileNetV2_Waste')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_resnet50v2(num_classes=NUM_CLASSES, learning_rate=1e-4, weight_decay=1e-4):
    """
    ResNet50V2 with frozen ImageNet base and custom 6-class head.
    Deep CNN baseline — ~25M parameters.
    """
    base = tf.keras.applications.ResNet50V2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False   # Phase 1: head only

    model = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(
            num_classes, activation='softmax',
            kernel_regularizer=tf.keras.regularizers.l2(weight_decay)
        )
    ], name='ResNet50V2_Waste')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("CNN model builders ready")

# ============================================================
# STEP 3B: ViT MODEL — LOAD PRETRAINED WEIGHTS FROM HUGGINGFACE
# ============================================================

In [ ]:
print("\n" + "=" * 60)
print("STEP 3B: LOADING ViT PRETRAINED WEIGHTS FROM HUGGINGFACE")
print("=" * 60)

pt_model   = ViTModel.from_pretrained('google/vit-base-patch16-224')
pt_model.eval()
pt_weights = {name: param.detach().numpy()
              for name, param in pt_model.named_parameters()}

print(f"Loaded {len(pt_weights)} weight tensors from google/vit-base-patch16-224")

In [ ]:
print("\n" + "=" * 60)
print("STEP 3C: ViT BUILDING BLOCKS")
print("=" * 60)

def mlp_block(x, hidden_dim, out_dim, name):
    """Two-layer MLP with GELU activation inside each transformer block."""
    x = keras.layers.Dense(hidden_dim, activation='gelu', name=f"{name}_fc1")(x)
    x = keras.layers.Dense(out_dim,    name=f"{name}_fc2")(x)
    return x


def attention_block(x, num_heads, head_dim, embed_dim, name):
    """
    Multi-Head Self-Attention.
    Every token (patch + CLS) attends to every other token simultaneously.
    This gives ViT a global receptive field from layer 1 — unlike CNNs.
    """
    # Project to Q, K, V
    q = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_q")(x)
    k = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_k")(x)
    v = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_v")(x)

    # Reshape to [batch, seq_len, num_heads, head_dim]
    q = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_q_reshape")(q)
    k = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_k_reshape")(k)
    v = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_v_reshape")(v)

    # Transpose to [batch, num_heads, seq_len, head_dim]
    q = keras.layers.Permute((2, 1, 3), name=f"{name}_q_perm")(q)
    k = keras.layers.Permute((2, 1, 3), name=f"{name}_k_perm")(k)
    v = keras.layers.Permute((2, 1, 3), name=f"{name}_v_perm")(v)

    scale = head_dim ** -0.5
    attn  = keras.layers.Lambda(
        lambda qk: keras.ops.softmax(
            keras.ops.matmul(qk[0],
                keras.ops.transpose(qk[1], (0, 1, 3, 2))) * scale,
            axis=-1
        ), name=f"{name}_attn_scores"
    )([q, k])

    out = keras.layers.Lambda(
        lambda av: keras.ops.matmul(av[0], av[1]),
        name=f"{name}_attn_out"
    )([attn, v])

    out = keras.layers.Permute((2, 1, 3), name=f"{name}_out_perm")(out)
    out = keras.layers.Reshape((-1, embed_dim),   name=f"{name}_out_reshape")(out)
    out = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_proj")(out)
    return out


def transformer_block(x, num_heads=12, embed_dim=768, mlp_dim=3072, name="block"):
    """
    Single ViT Transformer Encoder Block:
      LayerNorm → Multi-Head Attention → residual add
      LayerNorm → MLP → residual add
    """
    y = keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln1")(x)
    y = attention_block(y, num_heads=num_heads,
                        head_dim=embed_dim // num_heads,
                        embed_dim=embed_dim, name=f"{name}_attn")
    x = keras.layers.Add()([x, y])

    y = keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln2")(x)
    y = mlp_block(y, hidden_dim=mlp_dim, out_dim=embed_dim, name=f"{name}_mlp")
    x = keras.layers.Add()([x, y])
    return x

print("ViT building blocks ready")

In [ ]:
print("\n" + "=" * 60)
print("STEP 3D: BUILDING ViT KERAS MODEL")
print("=" * 60)

def build_vit_model(num_classes=NUM_CLASSES, learning_rate=2e-5, weight_decay=0.01):
    """
    ViT-base-patch16-224 implemented in Keras.
    Architecture:
      Image → 196 patch embeddings → [CLS]+patches → 12 transformer blocks
           → CLS token → Dense(6, softmax)

    Key advantage over CNNs: global self-attention from layer 1 allows
    the model to relate distant image regions (e.g. bottle shape vs label),
    resolving the texture-similarity problem (glass vs PET plastic).
    """
    num_patches = (IMG_SIZE // 16) ** 2   # 14×14 = 196
    embed_dim   = 768
    num_heads   = 12
    mlp_dim     = 3072
    num_layers  = 12

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="pixel_values")

    # ── Patch Embedding ──────────────────────────────────────
    # Conv2D with kernel=16, stride=16 extracts 196 non-overlapping patches
    patches = keras.layers.Conv2D(
        filters=embed_dim, kernel_size=16, strides=16,
        padding='valid', use_bias=True, name="patch_embed_proj"
    )(inputs)
    patches = keras.layers.Reshape(
        (num_patches, embed_dim), name="patch_embed_reshape"
    )(patches)

    # ── CLS Token ────────────────────────────────────────────
    # Learnable token prepended to patch sequence.
    # Aggregates global information across all 196 patches through attention.
    class CLSToken(keras.layers.Layer):
        def build(self, input_shape):
            self.cls = self.add_weight(
                name="cls", shape=(1, 1, embed_dim),
                initializer="zeros", trainable=True
            )
        def call(self, x):
            return keras.ops.broadcast_to(
                self.cls, (keras.ops.shape(x)[0], 1, embed_dim)
            )

    cls_tokens = CLSToken(name="cls_token")(patches)
    x = keras.layers.Concatenate(axis=1, name="cls_concat")([cls_tokens, patches])

    # ── Positional Embeddings ─────────────────────────────────
    # Learnable position encoding so the model knows spatial order of patches
    class PosEmbed(keras.layers.Layer):
        def build(self, input_shape):
            self.pos = self.add_weight(
                name="pos_embed",
                shape=(1, num_patches + 1, embed_dim),
                initializer="zeros", trainable=True
            )
        def call(self, x):
            return x + self.pos

    x = PosEmbed(name="pos_embed")(x)

    # ── 12 Transformer Encoder Blocks ────────────────────────
    for i in range(num_layers):
        x = transformer_block(
            x, num_heads=num_heads,
            embed_dim=embed_dim, mlp_dim=mlp_dim,
            name=f"block_{i}"
        )

    # ── Layer Norm + Extract CLS Token ───────────────────────
    x = keras.layers.LayerNormalization(epsilon=1e-6, name="ln_post")(x)
    # CLS token is at position 0 — contains global image summary
    cls_output = keras.layers.Lambda(
        lambda t: t[:, 0, :], name="cls_extract"
    )(x)

    # ── Classification Head ───────────────────────────────────
    cls_output = keras.layers.Dropout(0.1)(cls_output)
    outputs    = keras.layers.Dense(
        num_classes, name="classifier"
    )(cls_output)   # logits — SparseCategoricalCrossentropy handles softmax

    model = keras.Model(inputs=inputs, outputs=outputs, name="ViT_Waste")

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    return model

print("ViT model builder ready")

---
## STEP 3E: CROSS-VALIDATION FUNCTION

In [ ]:
print("\nLoading HuggingFace weights into Keras ViT model...")

vit_model = build_vit_model()

# Map HuggingFace parameter names → Keras layer names and set weights
def load_vit_weights(keras_model, pt_weights):
    """
    Transfer weights from PyTorch HuggingFace ViT into the Keras model.
    Layer names are matched by convention.
    """
    # Patch embedding
    if 'embeddings.patch_embeddings.projection.weight' in pt_weights:
        w = pt_weights['embeddings.patch_embeddings.projection.weight']
        b = pt_weights['embeddings.patch_embeddings.projection.bias']
        # PyTorch: [out, in, kH, kW] → TF: [kH, kW, in, out]
        w = np.transpose(w, (2, 3, 1, 0))
        keras_model.get_layer("patch_embed_proj").set_weights([w, b])

    # Positional embeddings
    if 'embeddings.position_embeddings' in pt_weights:
        pos = pt_weights['embeddings.position_embeddings']
        keras_model.get_layer("pos_embed").set_weights([pos])

    # Transformer blocks
    for i in range(12):
        prefix = f"encoder.layer.{i}"
        try:
            # Layer norms
            keras_model.get_layer(f"block_{i}_ln1").set_weights([
                pt_weights[f"{prefix}.layernorm_before.weight"],
                pt_weights[f"{prefix}.layernorm_before.bias"]
            ])
            keras_model.get_layer(f"block_{i}_ln2").set_weights([
                pt_weights[f"{prefix}.layernorm_after.weight"],
                pt_weights[f"{prefix}.layernorm_after.bias"]
            ])
            # Attention Q, K, V projections
            for proj in ['query', 'key', 'value']:
                short = proj[0]  # q, k, v
                w = pt_weights[f"{prefix}.attention.attention.{proj}.weight"].T
                b = pt_weights[f"{prefix}.attention.attention.{proj}.bias"]
                keras_model.get_layer(f"block_{i}_attn_{short}").set_weights([w, b])
            # Attention output projection
            w = pt_weights[f"{prefix}.attention.output.dense.weight"].T
            b = pt_weights[f"{prefix}.attention.output.dense.bias"]
            keras_model.get_layer(f"block_{i}_attn_proj").set_weights([w, b])
            # MLP
            w1 = pt_weights[f"{prefix}.intermediate.dense.weight"].T
            b1 = pt_weights[f"{prefix}.intermediate.dense.bias"]
            keras_model.get_layer(f"block_{i}_mlp_fc1").set_weights([w1, b1])
            w2 = pt_weights[f"{prefix}.output.dense.weight"].T
            b2 = pt_weights[f"{prefix}.output.dense.bias"]
            keras_model.get_layer(f"block_{i}_mlp_fc2").set_weights([w2, b2])
        except Exception as e:
            print(f"  Block {i} weight transfer note: {e}")

    # Final layer norm
    try:
        keras_model.get_layer("ln_post").set_weights([
            pt_weights['layernorm.weight'],
            pt_weights['layernorm.bias']
        ])
    except Exception as e:
        print(f"  Final LN note: {e}")

    print("Weight transfer complete")

load_vit_weights(vit_model, pt_weights)

# ============================================================
# STEP 3E: CROSS-VALIDATION FUNCTION
# ============================================================

---
## STEP 4: PHASE 1 — 5-FOLD CROSS-VALIDATION ON 80% POOL

In [ ]:
print("\n" + "=" * 60)
print("STEP 3E: 5-FOLD CROSS-VALIDATION FUNCTION")
print("=" * 60)

def measure_latency(model, dataset, num_samples=50):
    """Measure mean inference latency in ms per single image."""
    latencies = []
    for images, _ in dataset.take(1):
        batch = images[:min(num_samples, tf.shape(images)[0])]
        for i in range(len(batch)):
            single = batch[i:i+1]
            start  = time.perf_counter()
            model.predict(single, verbose=0)
            latencies.append((time.perf_counter() - start) * 1000)
    return float(np.mean(latencies))


def cross_validate_model(model_builder, model_name,
                          paths, label_list,
                          batch_size, is_vit=False,
                          num_folds=5, epochs=20):
    """
    Run stratified k-fold cross-validation on the 80% training pool.

    For each fold:
      - Train on 4 folds (~64% of total data)
      - Validate on 1 fold (~16% of total data)
      - Record: Macro F1, Accuracy, Precision, Recall, Latency

    Returns dict of lists with one value per fold.
    """
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

    results = defaultdict(list)

    for fold, (train_idx, val_idx) in enumerate(skf.split(paths, label_list)):
        print(f"\n{'='*50}")
        print(f"  {model_name} — Fold {fold+1}/{num_folds}")
        print(f"{'='*50}")

        fold_train_paths  = [paths[i]      for i in train_idx]
        fold_train_labels = [label_list[i] for i in train_idx]
        fold_val_paths    = [paths[i]      for i in val_idx]
        fold_val_labels   = [label_list[i] for i in val_idx]

        print(f"  Train: {len(fold_train_paths)} | Val: {len(fold_val_paths)}")

        # Build dataset pipelines
        if is_vit:
            train_ds = create_vit_dataset(fold_train_paths, fold_train_labels,
                                           batch_size, augment=True, shuffle=True)
            val_ds   = create_vit_dataset(fold_val_paths,   fold_val_labels,
                                           batch_size, augment=False)
        else:
            train_ds = create_cnn_dataset(fold_train_paths, fold_train_labels,
                                           batch_size, augment=True, shuffle=True)
            val_ds   = create_cnn_dataset(fold_val_paths,   fold_val_labels,
                                           batch_size, augment=False)

        # Build a fresh model for each fold
        model = model_builder()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5,
                restore_best_weights=True, verbose=0
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3,
                min_lr=1e-7, verbose=0
            )
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )

        # Collect predictions
        y_true, y_pred = [], []
        for images, lbl in val_ds:
            preds = model.predict(images, verbose=0)
            if is_vit:
                y_true.extend(lbl.numpy().tolist())
                y_pred.extend(np.argmax(preds, axis=1).tolist())
            else:
                y_true.extend(np.argmax(lbl.numpy(), axis=1).tolist())
                y_pred.extend(np.argmax(preds, axis=1).tolist())

        # Compute metrics
        macro_f1  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
        accuracy  = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall    = recall_score(y_true, y_pred,    average='macro', zero_division=0)
        latency   = measure_latency(model, val_ds)

        results['macro_f1'].append(macro_f1)
        results['accuracy'].append(accuracy)
        results['precision'].append(precision)
        results['recall'].append(recall)
        results['latency_ms'].append(latency)
        results['history'].append(history)

        print(f"  Fold {fold+1}: F1={macro_f1:.4f} | Acc={accuracy:.4f} | Latency={latency:.1f}ms")

    # Summary
    print(f"\n{'='*50}")
    print(f"  {model_name} — 5-Fold CV Summary")
    print(f"{'='*50}")
    print(f"  Macro F1  : {np.mean(results['macro_f1']):.4f} ± {np.std(results['macro_f1']):.4f}")
    print(f"  Accuracy  : {np.mean(results['accuracy']):.4f} ± {np.std(results['accuracy']):.4f}")
    print(f"  Precision : {np.mean(results['precision']):.4f} ± {np.std(results['precision']):.4f}")
    print(f"  Recall    : {np.mean(results['recall']):.4f} ± {np.std(results['recall']):.4f}")
    print(f"  Latency   : {np.mean(results['latency_ms']):.2f} ± {np.std(results['latency_ms']):.2f} ms")

    return dict(results)

print("Cross-validation function ready")

# ============================================================
# STEP 4: PHASE 1 — 5-FOLD CROSS-VALIDATION ON 80% POOL
# ============================================================

In [ ]:
print("\n" + "=" * 60)
print("STEP 4: PHASE 1 — RUNNING 5-FOLD CROSS-VALIDATION")
print("=" * 60)

# MobileNetV2
print("\n>>> MobileNetV2")
mobilenet_cv = cross_validate_model(
    lambda: build_mobilenetv2(),
    "MobileNetV2",
    train_paths, train_labels,
    batch_size=32, is_vit=False
)

# ResNet50V2
print("\n>>> ResNet50V2")
resnet_cv = cross_validate_model(
    lambda: build_resnet50v2(),
    "ResNet50V2",
    train_paths, train_labels,
    batch_size=32, is_vit=False
)

# ViT
print("\n>>> ViT-base-patch16-224")
vit_cv = cross_validate_model(
    lambda: build_vit_model(),
    "ViT",
    train_paths, train_labels,
    batch_size=16, is_vit=True
)

---
## STEP 5: PHASE 2 — FINAL TRAINING ON 80% POOL

In [ ]:
print("\n" + "=" * 60)
print("STEP 4 RESULTS: PHASE 1 CV SUMMARY")
print("=" * 60)

cv_summary = pd.DataFrame({
    'Model': ['MobileNetV2', 'ResNet50V2', 'ViT'],
    'Macro F1 (mean±std)': [
        f"{np.mean(mobilenet_cv['macro_f1']):.4f} ± {np.std(mobilenet_cv['macro_f1']):.4f}",
        f"{np.mean(resnet_cv['macro_f1']):.4f}    ± {np.std(resnet_cv['macro_f1']):.4f}",
        f"{np.mean(vit_cv['macro_f1']):.4f}       ± {np.std(vit_cv['macro_f1']):.4f}",
    ],
    'Accuracy': [
        f"{np.mean(mobilenet_cv['accuracy']):.4f}",
        f"{np.mean(resnet_cv['accuracy']):.4f}",
        f"{np.mean(vit_cv['accuracy']):.4f}",
    ],
    'Latency (ms)': [
        f"{np.mean(mobilenet_cv['latency_ms']):.2f}",
        f"{np.mean(resnet_cv['latency_ms']):.2f}",
        f"{np.mean(vit_cv['latency_ms']):.2f}",
    ]
})
print(cv_summary.to_string(index=False))
cv_summary.to_csv('phase1_cv_results.csv', index=False)

# Save raw results for analysis
with open('phase1_raw_results.pkl', 'wb') as f:
    pickle.dump({'mobilenet': mobilenet_cv, 'resnet': resnet_cv, 'vit': vit_cv}, f)
print("\nPhase 1 results saved")

# ============================================================
# STEP 5: PHASE 2 — FINAL TRAINING ON 80% POOL
# ============================================================

In [ ]:
print("\n" + "=" * 60)
print("STEP 5A: CREATING 90/10 FINAL TRAIN/VAL SPLIT")
print("=" * 60)

train_final, val_final, train_labels_final, val_labels_final = train_test_split(
    train_paths, train_labels,
    test_size=0.1,
    stratify=train_labels,
    random_state=SEED
)

print(f"Final Train (90% of 80%): {len(train_final)} images")
print(f"Final Val  (10% of 80%): {len(val_final)} images")
print_dist(train_labels_final, "Final Train")
print_dist(val_labels_final,   "Final Val")

In [ ]:
print("\n" + "=" * 60)
print("STEP 5B: FINAL TRAINING — MobileNetV2")
print("=" * 60)

MOBILENET_CFG = {'lr': 1e-4, 'batch': 32, 'wd': 1e-4}

train_ds_mn = create_cnn_dataset(train_final, train_labels_final,
                                  MOBILENET_CFG['batch'], augment=True, shuffle=True)
val_ds_mn   = create_cnn_dataset(val_final,   val_labels_final,
                                  MOBILENET_CFG['batch'], augment=False)

mobilenet_final = build_mobilenetv2(learning_rate=MOBILENET_CFG['lr'],
                                     weight_decay=MOBILENET_CFG['wd'])

callbacks_final = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'mobilenet_final_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

history_mn = mobilenet_final.fit(
    train_ds_mn, validation_data=val_ds_mn,
    epochs=30, callbacks=callbacks_final, verbose=1
)
mobilenet_final.save('mobilenet_final_model.keras')
print("MobileNetV2 final model saved")

In [ ]:
print("\n" + "=" * 60)
print("STEP 5C: FINAL TRAINING — ResNet50V2")
print("=" * 60)

RESNET_CFG = {'lr': 1e-4, 'batch': 32, 'wd': 1e-4}

train_ds_rn = create_cnn_dataset(train_final, train_labels_final,
                                  RESNET_CFG['batch'], augment=True, shuffle=True)
val_ds_rn   = create_cnn_dataset(val_final,   val_labels_final,
                                  RESNET_CFG['batch'], augment=False)

resnet_final = build_resnet50v2(learning_rate=RESNET_CFG['lr'],
                                 weight_decay=RESNET_CFG['wd'])

callbacks_final[2] = tf.keras.callbacks.ModelCheckpoint(
    'resnet_final_best.keras', monitor='val_accuracy',
    save_best_only=True, verbose=1
)

history_rn = resnet_final.fit(
    train_ds_rn, validation_data=val_ds_rn,
    epochs=30, callbacks=callbacks_final, verbose=1
)
resnet_final.save('resnet_final_model.keras')
print("ResNet50V2 final model saved")

In [ ]:
print("\n" + "=" * 60)
print("STEP 5D: FINAL TRAINING — ViT (Two-Stage Fine-Tune)")
print("=" * 60)

VIT_CFG = {'lr': 2e-5, 'batch': 16, 'wd': 0.01}

train_ds_vit = create_vit_dataset(train_final, train_labels_final,
                                   VIT_CFG['batch'], augment=True, shuffle=True)
val_ds_vit   = create_vit_dataset(val_final,   val_labels_final,
                                   VIT_CFG['batch'], augment=False)

vit_final = build_vit_model(learning_rate=VIT_CFG['lr'],
                              weight_decay=VIT_CFG['wd'])
load_vit_weights(vit_final, pt_weights)

callbacks_final[2] = tf.keras.callbacks.ModelCheckpoint(
    'vit_stage1_best.keras', monitor='val_accuracy',
    save_best_only=True, verbose=1
)

# Stage 1: Frozen backbone — train classification head only
print("\n[Stage 1] Frozen backbone — training head only...")
history_vit_s1 = vit_final.fit(
    train_ds_vit, validation_data=val_ds_vit,
    epochs=30, callbacks=callbacks_final, verbose=1
)
vit_final.save('vit_stage1_model.keras')

# Stage 2: Unfreeze last 50% of transformer blocks
print("\n[Stage 2] Unfreezing last 50% of transformer blocks...")
transformer_blocks = [l for l in vit_final.layers if l.name.startswith("block_")]
num_to_unfreeze    = int(len(transformer_blocks) * 0.5)

for blk in transformer_blocks[-num_to_unfreeze:]:
    blk.trainable = True

print(f"Unfroze {num_to_unfreeze}/{len(transformer_blocks)} blocks")

vit_final.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

callbacks_final[2] = tf.keras.callbacks.ModelCheckpoint(
    'vit_final_best.keras', monitor='val_accuracy',
    save_best_only=True, verbose=1
)

history_vit_s2 = vit_final.fit(
    train_ds_vit, validation_data=val_ds_vit,
    epochs=20, callbacks=callbacks_final, verbose=1
)
vit_final.save('vit_final_model.keras')
print("ViT final model saved")

---
## STEP 6: FINAL TEST EVALUATION — ONCE ON ISOLATED 20% SET

In [ ]:
def plot_training_curves(history, model_name, save_path=None):
    """Plot training vs validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
    ax1.set_title(f'{model_name} — Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'],     label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
    ax2.set_title(f'{model_name} — Loss', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

os.makedirs('outputs/curves', exist_ok=True)
plot_training_curves(history_mn,    "MobileNetV2", "outputs/curves/mobilenet_curves.png")
plot_training_curves(history_rn,    "ResNet50V2",  "outputs/curves/resnet_curves.png")
plot_training_curves(history_vit_s2,"ViT Stage 2", "outputs/curves/vit_curves.png")

# ============================================================
# STEP 6: FINAL TEST EVALUATION — ONCE ON ISOLATED 20% SET
# ============================================================

In [ ]:
print("\n" + "=" * 60)
print("STEP 6: FINAL TEST EVALUATION — ISOLATED 20% TEST SET")
print("=" * 60)

with open('test_set_isolated.pkl', 'rb') as f:
    test_data = pickle.load(f)

test_paths_eval  = test_data['paths']
test_labels_eval = test_data['labels']
print(f"Test set: {len(test_paths_eval)} images — evaluating ONCE")

test_ds_cnn = create_cnn_dataset(test_paths_eval, test_labels_eval,
                                  batch_size=32, augment=False)
test_ds_vit = create_vit_dataset(test_paths_eval, test_labels_eval,
                                  batch_size=16, augment=False)


def evaluate_on_test(model, model_name, test_ds, is_vit=False):
    """Run inference on test set and return all metrics and predictions."""
    y_true, y_pred = [], []
    for images, lbl in test_ds:
        preds = model.predict(images, verbose=0)
        if is_vit:
            y_true.extend(lbl.numpy().tolist())
            y_pred.extend(np.argmax(preds, axis=1).tolist())
        else:
            y_true.extend(np.argmax(lbl.numpy(), axis=1).tolist())
            y_pred.extend(np.argmax(preds, axis=1).tolist())

    macro_f1 = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    latency  = measure_latency(model, test_ds)

    print(f"\n{'='*50}")
    print(f"{model_name} — Test Set Results")
    print(f"{'='*50}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    print(f"Macro F1 : {macro_f1:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Latency  : {latency:.2f} ms/image")

    return {'macro_f1': macro_f1, 'accuracy': accuracy,
            'latency': latency, 'y_true': y_true, 'y_pred': y_pred}


mn_test  = evaluate_on_test(mobilenet_final, "MobileNetV2", test_ds_cnn, is_vit=False)
rn_test  = evaluate_on_test(resnet_final,    "ResNet50V2",  test_ds_cnn, is_vit=False)
vit_test = evaluate_on_test(vit_final,       "ViT",         test_ds_vit, is_vit=True)

# Save results
with open('final_test_results.pkl', 'wb') as f:
    pickle.dump({'mobilenet': mn_test, 'resnet': rn_test, 'vit': vit_test}, f)

# Final comparison table
comparison_df = pd.DataFrame({
    'Model':      ['MobileNetV2', 'ResNet50V2', 'ViT'],
    'Macro F1':   [f"{mn_test['macro_f1']:.4f}",
                   f"{rn_test['macro_f1']:.4f}",
                   f"{vit_test['macro_f1']:.4f}"],
    'Accuracy':   [f"{mn_test['accuracy']:.4f}",
                   f"{rn_test['accuracy']:.4f}",
                   f"{vit_test['accuracy']:.4f}"],
    'Latency(ms)':[f"{mn_test['latency']:.2f}",
                   f"{rn_test['latency']:.2f}",
                   f"{vit_test['latency']:.2f}"],
})
print("\nFINAL COMPARISON TABLE:")
print(comparison_df.to_string(index=False))
comparison_df.to_csv('final_comparison_results.csv', index=False)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, save_path=None):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{model_name} — Confusion Matrix (Normalised)', fontweight='bold')
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

os.makedirs('outputs/matrices', exist_ok=True)
plot_confusion_matrix(mn_test['y_true'],  mn_test['y_pred'],  "MobileNetV2",
                      "outputs/matrices/mobilenet_confusion.png")
plot_confusion_matrix(rn_test['y_true'],  rn_test['y_pred'],  "ResNet50V2",
                      "outputs/matrices/resnet_confusion.png")
plot_confusion_matrix(vit_test['y_true'], vit_test['y_pred'], "ViT",
                      "outputs/matrices/vit_confusion.png")

---
## STEP 7: EXPLAINABILITY — ATTENTION MAPS & GRAD-CAM

In [ ]:
print("\n" + "=" * 60)
print("STEP 6: McNEMAR'S STATISTICAL SIGNIFICANCE TEST")
print("=" * 60)

def run_mcnemar(name_a, pred_a, name_b, pred_b, y_true):
    """
    McNemar's test compares two classifiers on the same test set.
    H0: both models make errors on the same images.
    p < 0.05 = the difference is statistically significant.
    """
    correct_a = np.array(pred_a) == np.array(y_true)
    correct_b = np.array(pred_b) == np.array(y_true)

    b = int(np.sum(correct_a & ~correct_b))   # A right, B wrong
    c = int(np.sum(~correct_a & correct_b))   # A wrong, B right

    result = mcnemar([[0, b], [c, 0]], exact=True)

    sig = "✓ SIGNIFICANT" if result.pvalue < 0.05 else "✗ Not significant"
    print(f"\n{name_a} vs {name_b}:")
    print(f"  b={b} ({name_a} right, {name_b} wrong)")
    print(f"  c={c} ({name_a} wrong, {name_b} right)")
    print(f"  p-value = {result.pvalue:.6f}  {sig}")
    return result.pvalue

y_true = vit_test['y_true']
run_mcnemar("ViT", vit_test['y_pred'], "ResNet50V2",  rn_test['y_pred'],  y_true)
run_mcnemar("ViT", vit_test['y_pred'], "MobileNetV2", mn_test['y_pred'],  y_true)

# ============================================================
# STEP 7: EXPLAINABILITY — ATTENTION MAPS & GRAD-CAM
# ============================================================

In [ ]:
print("\n" + "=" * 60)
print("STEP 7A: GRAD-CAM — CNN EXPLAINABILITY")
print("=" * 60)

def get_gradcam(model, image, class_idx, last_conv_layer_name):
    """
    Compute Grad-CAM saliency map.
    Highlights which spatial regions of the image most influenced
    the predicted class — shows LOCAL texture focus of CNNs.
    """
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    img_batch = tf.expand_dims(image, axis=0)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch, training=False)
        loss = preds[:, class_idx]

    grads        = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    cam = conv_out[0] @ pooled_grads[..., tf.newaxis]
    cam = tf.squeeze(cam)
    cam = tf.nn.relu(cam).numpy()

    import cv2
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

In [ ]:
print("\n" + "=" * 60)
print("STEP 7B: ViT ATTENTION MAP EXTRACTION")
print("=" * 60)

def get_attention_map(image_np):
    """
    Extract attention weights from the last ViT encoder block.
    CLS token attention over 196 patches → reshaped to 14x14 → upsampled to 224x224.
    Shows GLOBAL attention — what the ViT considered important across the whole image.
    """
    import cv2
    img_tensor = torch.tensor(image_np).permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        outputs = pt_model(
            pixel_values=img_tensor,
            output_attentions=True
        )

    # Last block attention: [1, 12, 197, 197]
    last_attn  = outputs.attentions[-1]
    avg_attn   = last_attn.mean(dim=1)[0]     # [197, 197]
    cls_attn   = avg_attn[0, 1:].numpy()      # [196] — CLS → patches
    attn_map   = cls_attn.reshape(14, 14)
    attn_map   = cv2.resize(attn_map, (IMG_SIZE, IMG_SIZE))
    attn_map   = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map

---
## FINAL SUMMARY

In [ ]:
print("\n" + "=" * 60)
print("STEP 7C: GENERATING XAI HEATMAP GALLERY")
print("=" * 60)

import cv2

os.makedirs('outputs/xai_gallery', exist_ok=True)

def overlay(image, heatmap, alpha=0.5):
    """Blend coloured heatmap onto original image."""
    colormap    = plt.cm.jet(heatmap)[..., :3]
    return np.clip((1 - alpha) * image + alpha * colormap, 0, 1)


def generate_xai_panel(image_path_rel, true_label, save_name):
    """
    Generate a 4-panel XAI comparison figure for one image:
    Original | MobileNet Grad-CAM | ResNet Grad-CAM | ViT Attention Map
    """
    full_path = os.path.join(DATA_DIR, image_path_rel)
    img       = tf.io.read_file(full_path)
    img       = tf.image.decode_jpeg(img, channels=3)
    img       = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img_norm  = (tf.cast(img, tf.float32) / 255.0).numpy()

    # Grad-CAM for CNNs
    img_batch     = np.expand_dims(img_norm, 0)
    mn_pred       = mobilenet_final.predict(img_batch, verbose=0)[0]
    rn_pred       = resnet_final.predict(img_batch, verbose=0)[0]
    mn_class      = np.argmax(mn_pred)
    rn_class      = np.argmax(rn_pred)

    mn_cam  = get_gradcam(mobilenet_final, img_norm, mn_class, "Conv_1")
    rn_cam  = get_gradcam(resnet_final,    img_norm, rn_class, "conv5_block3_out")

    # ViT attention map
    vit_attn = get_attention_map(img_norm)

    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"True: {true_label}  |  MN: {CLASS_NAMES[mn_class]}  "
                 f"|  RN: {CLASS_NAMES[rn_class]}  "
                 f"|  ViT: {CLASS_NAMES[np.argmax(vit_final.predict(img_batch, verbose=0)[0])]}",
                 fontsize=12, fontweight='bold')

    panels = [
        (img_norm,                "Original Image"),
        (overlay(img_norm, mn_cam), f"MobileNetV2 Grad-CAM\n(Local texture focus)"),
        (overlay(img_norm, rn_cam), f"ResNet50V2 Grad-CAM\n(Local texture focus)"),
        (overlay(img_norm, vit_attn), f"ViT Attention Map\n(Global spatial focus)"),
    ]

    for ax, (panel, title) in zip(axes, panels):
        ax.imshow(panel)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'outputs/xai_gallery/{save_name}', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/xai_gallery/{save_name}")


# Generate gallery for 2 images from each class
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_images = [p for p, l in zip(test_paths_eval, test_labels_eval) if l == cls_idx]
    for j, img_path in enumerate(cls_images[:2]):
        generate_xai_panel(img_path, cls_name, f"{cls_name}_{j+1}_xai.png")


print("\n" + "=" * 60)
print("PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 60)

print("\nPhase 1 — 5-Fold Cross-Validation:")
print(cv_summary.to_string(index=False))

print("\nPhase 2 — Final Test Set Results:")
print(comparison_df.to_string(index=False))

print("\nFiles saved:")
saved_files = [
    "test_set_isolated.pkl",
    "phase1_cv_results.csv",
    "phase1_raw_results.pkl",
    "mobilenet_final_model.keras",
    "resnet_final_model.keras",
    "vit_final_model.keras",
    "final_test_results.pkl",
    "final_comparison_results.csv",
    "outputs/curves/          ← training curves",
    "outputs/matrices/        ← confusion matrices",
    "outputs/xai_gallery/     ← attention maps vs Grad-CAM",
]
for f in saved_files:
    print(f"  {f}")